<a href="https://colab.research.google.com/github/langchain-samples/lc-colab-workshops/blob/main/notebooks/10_evals_from_traces.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 10 · From vibes to a test suite

Part 1 built agents. Every single one you judged the same way: you read the output and decided
it looked about right.

That does not scale past a handful of examples, and it tells you nothing about whether
tomorrow's prompt change breaks today's behaviour. Part 2 replaces it with a test suite — and
like most good test suites, it starts from **one real bug**.

**New in this lesson:** finding a failure in a trace, turning it into a dataset, and running
your first `evaluate()`.

> **Need a key?** You need a LangSmith API key stored in Colab Secrets (🔑 in the left
> sidebar) as `LANGSMITH_API_KEY`, with **"Notebook access" turned on**. If you have not done
> that yet, run **[00 · Setup](https://colab.research.google.com/github/langchain-samples/lc-colab-workshops/blob/main/notebooks/00_setup.ipynb)** first — it takes 10 minutes and
> checks everything.

In [ ]:
# --- snippet:setup v1 ---
%pip install -qq \
  "deepagents~=0.7.6" \
  "langchain~=1.3.15" \
  "langchain-openai~=1.5.1" \
  "langsmith~=0.11.0"

import os

try:
    from google.colab import userdata

    key = userdata.get("LANGSMITH_API_KEY")
except Exception:  # not on Colab, or secret unavailable
    from getpass import getpass

    key = os.environ.get("LANGSMITH_API_KEY") or getpass("LANGSMITH_API_KEY: ")

os.environ["LANGSMITH_API_KEY"] = key
os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_PROJECT"] = "lcw-10-evals"

# One constant, used everywhere. Models are served by the LangSmith gateway,
# so this key is the only credential the notebook needs.
MODEL = "langsmith:openai/gpt-5.6-luna"
# --- /snippet ---

print("Ready.")

In [ ]:
#@title Synthetic support data (run me) { display-mode: "form" }
# Six orders, eight tickets, one refund policy. Small on purpose: you should be able to
# read the whole dataset and judge the agent's answers yourself.

# --- snippet:support_data v1 ---
ORDERS = [
    {"id": "1042", "customer": "avery@example.com", "item": "Standing desk", "status": "delivered",  "days_ago": 3,  "price": 429.00},
    {"id": "1043", "customer": "jordan@example.com", "item": "Desk lamp",     "status": "delivered",  "days_ago": 45, "price": 39.00},
    {"id": "1044", "customer": "avery@example.com", "item": "Monitor arm",    "status": "in_transit", "days_ago": 1,  "price": 89.00},
    {"id": "1045", "customer": "sam@example.com",   "item": "Office chair",   "status": "delivered",  "days_ago": 10, "price": 249.00},
    {"id": "1046", "customer": "riley@example.com", "item": "Keyboard tray",  "status": "cancelled",  "days_ago": 7,  "price": 59.00},
    {"id": "1047", "customer": "sam@example.com",   "item": "Laptop stand",   "status": "delivered",  "days_ago": 62, "price": 45.00},
]

TICKETS = [
    {"id": "T-1", "order_id": "1042", "text": "Desk arrived with a cracked leg. Photos attached."},
    {"id": "T-2", "order_id": "1043", "text": "Lamp stopped working. Bought it over a month ago."},
    {"id": "T-3", "order_id": "1044", "text": "Where is my monitor arm? Ordered yesterday."},
    {"id": "T-4", "order_id": "1045", "text": "Chair is fine but I ordered the wrong colour. Can I swap?"},
    {"id": "T-5", "order_id": "1046", "text": "I cancelled this but was still charged."},
    {"id": "T-6", "order_id": "1047", "text": "Laptop stand wobbles. Had it two months."},
    {"id": "T-7", "order_id": "1042", "text": "Following up on the cracked desk leg. Any update?"},
    {"id": "T-8", "order_id": "9999", "text": "Order never arrived."},
]

REFUND_POLICY = """
# Refund policy

- Damaged on arrival: full refund or replacement, no time limit. Photos required.
- Faulty within 30 days of delivery: full refund or replacement.
- Faulty after 30 days: repair only. No refund.
- Wrong item ordered by the customer: exchange within 14 days of delivery. Restocking fee 10%.
- Cancelled orders: refund within 5 business days. Escalate if the customer was charged.
- Refunds above $200 require human approval.
"""
# --- /snippet ---

print(f"{len(ORDERS)} orders, {len(TICKETS)} tickets, {len(REFUND_POLICY.splitlines())} lines of policy")

In [ ]:
# --- snippet:support_agent v1 ---
from langchain_core.tools import tool


@tool
def lookup_order(order_id: str) -> str:
    """Look up a single order by its numeric ID.

    Returns the customer email, item, delivery status, days since order, and price.
    Use this before answering any question about a specific order.
    """
    for order in ORDERS:
        if order["id"] == order_id:
            return (
                f"Order {order['id']}: {order['item']}, ${order['price']:.2f}, "
                f"status={order['status']}, ordered {order['days_ago']} days ago, "
                f"customer={order['customer']}"
            )
    # An error message is an instruction to a reader who cannot see your code.
    return (
        f"No order with ID {order_id!r}. Order IDs are 4 digits (e.g. 1042). "
        f"Ask the customer to re-check the number on their confirmation email."
    )


@tool
def search_tickets(query: str) -> str:
    """Search past support tickets for a keyword.

    Use this to find whether a customer has written in before about the same problem.
    """
    hits = [t for t in TICKETS if query.lower() in t["text"].lower()]
    if not hits:
        return f"No tickets matching {query!r}."
    return "\n".join(f"{t['id']} (order {t['order_id']}): {t['text']}" for t in hits)


@tool
def get_refund_policy() -> str:
    """Return the full refund policy. Consult this before promising any refund."""
    return REFUND_POLICY
# --- /snippet ---

print("3 tools defined")

---

## 1. Generate some traffic

Six realistic support questions. One of them the agent gets wrong in a specific, instructive way.

In [ ]:
from deepagents import create_deep_agent

SUPPORT_PROMPT = (
    "You are a customer support agent for an office furniture retailer.\n"
    "Look up the order before answering questions about it.\n"
    "Check the refund policy before promising a refund, replacement, or exchange.\n"
    "Be concise and quote the policy line you relied on."
)

agent = create_deep_agent(
    model=MODEL,
    tools=[lookup_order, search_tickets, get_refund_policy],
    system_prompt=SUPPORT_PROMPT,
)

QUESTIONS = [
    "Order 1042 arrived with a cracked leg. What are the customer's options?",
    "Order 1047 - the laptop stand wobbles. Can they get a refund?",
    "Order 1043's desk lamp stopped working. What can we offer?",
    "Order 1045 - customer ordered the wrong colour chair. Options?",
    "Order 1046 was cancelled but the customer says they were charged.",
    "Order 1044 - where is it?",
]

runs = []
for q in QUESTIONS:
    out = agent.invoke({"messages": [{"role": "user", "content": q}]})
    runs.append((q, out["messages"][-1].text))
    print(f"Q: {q}\nA: {out['messages'][-1].text[:220]}\n{'-' * 70}")

---

## 2. Find the bad one

Read those answers against the policy you loaded at the top. Two are worth checking closely:

- **Order 1047** — laptop stand, delivered **62 days ago**. Policy: *"Faulty after 30 days:
  repair only. No refund."*
- **Order 1043** — desk lamp, delivered **45 days ago**. Same rule.

An agent that offers a refund on either has invented a policy that costs real money. It will
often sound *more* helpful while doing it, which is exactly what makes this class of bug
dangerous.

Now find it in LangSmith rather than by eye — that is the workflow that survives 6,000 runs.

> 📸 **`10-find-failure.png`** — The LangSmith project view for lcw-10-evals filtered to the six support runs, with one run selected and its final answer visible in the right-hand pane.
>
> *Caption:* Reading runs in the project view. This works at six runs and fails at six thousand.
>
> `https://raw.githubusercontent.com/langchain-samples/lc-colab-workshops/main/assets/screenshots/10-find-failure.png`

### The pivot

You just did QA by reading. It does not scale, and worse, it is not **repeatable** — nothing you
just did will tell you tomorrow whether a prompt change reintroduced the same bug.

A dataset is how you make a bug repeatable. It is a bug report you can re-run.

---

## 3. Three levels of eval

The whole of Part 2 hangs on this table. It maps directly onto testing you already do.

| Testing | Agent eval | The question | Lesson |
|---|---|---|---|
| Unit test | **single step** | did *this one step* do the right thing? | 11 |
| Integration test | **trajectory** | did the steps compose into a sane path? | 12 |
| End-to-end test | **final response** | did the user get a good answer? | this one |

The instinct is to write only the third kind, because it is the one that matches "did it work?".
The problem is that a final-response failure tells you *that* something broke and never *where*.

---

## 4. Build the dataset

In the UI you would add a bad run straight from its trace — **Add to Dataset**, then correct the
expected output.

> 📸 **`10-add-to-dataset.png`** — The LangSmith trace view with the 'Add to Dataset' dialog open, showing the run's inputs on the left and an editable reference output on the right.
>
> *Caption:* Promoting a failing run into a dataset, straight from its trace.
>
> `https://raw.githubusercontent.com/langchain-samples/lc-colab-workshops/main/assets/screenshots/10-add-to-dataset.png`

Here is the same thing from the SDK, which is what you will actually do in bulk. Note that the
reference outputs encode the **policy**, not a particular wording.

In [ ]:
from langsmith import Client

client = Client()

DATASET_NAME = "support-agent-v1"

EXAMPLES = [
    {"question": "Order 1042 arrived with a cracked leg. What are the customer's options?",
     "expected_outcome": "refund_or_replacement",
     "policy": "Damaged on arrival: full refund or replacement, no time limit. Photos required."},
    {"question": "Order 1047 - the laptop stand wobbles. Can they get a refund?",
     "expected_outcome": "repair_only",
     "policy": "Faulty after 30 days: repair only. No refund. (delivered 62 days ago)"},
    {"question": "Order 1043's desk lamp stopped working. What can we offer?",
     "expected_outcome": "repair_only",
     "policy": "Faulty after 30 days: repair only. No refund. (delivered 45 days ago)"},
    {"question": "Order 1045 - customer ordered the wrong colour chair. Options?",
     "expected_outcome": "exchange_with_fee",
     "policy": "Wrong item ordered by customer: exchange within 14 days. Restocking fee 10%."},
    {"question": "Order 1046 was cancelled but the customer says they were charged.",
     "expected_outcome": "escalate",
     "policy": "Cancelled orders: refund within 5 business days. Escalate if charged."},
    {"question": "Order 1044 - where is it?",
     "expected_outcome": "in_transit_no_refund",
     "policy": "Order is in transit, ordered 1 day ago. No refund question arises."},
    # A case nobody ran yet: the boundary between the two refund rules.
    {"question": "Order 1045's chair developed a fault today, 10 days after delivery. Refund?",
     "expected_outcome": "refund_or_replacement",
     "policy": "Faulty within 30 days of delivery: full refund or replacement."},
]

if client.has_dataset(dataset_name=DATASET_NAME):
    client.delete_dataset(dataset_name=DATASET_NAME)

dataset = client.create_dataset(
    dataset_name=DATASET_NAME,
    description="Support agent policy adherence, built from lesson 10 traces.",
)

client.create_examples(
    dataset_id=dataset.id,
    examples=[
        {
            "inputs": {"question": e["question"]},
            "outputs": {"expected_outcome": e["expected_outcome"], "policy": e["policy"]},
        }
        for e in EXAMPLES
    ],
)

print(f"{DATASET_NAME}: {len(EXAMPLES)} examples")
print(f"https://smith.langchain.com/datasets")

### 🧠 Checkpoint

Each example has an input and a reference output. But the reference outputs above are things
like `repair_only` and a policy line — not the sentence you expect the agent to say.

Why not just store the ideal answer text?

<details><summary>Show answer</summary>

Because there are thousands of correct wordings and you would be testing the wrong thing.

*"I'm afraid we can only offer a repair"* and *"Since it's been over 30 days, we can repair it
for you"* are both right. String-matching either one produces a test that fails on every
harmless rewording — and once a test fails for reasons nobody cares about, people stop trusting
it, and then stop reading it.

So you store the **decision** (`repair_only`) and the **reason** (the policy line). Those are
what must be true. How the agent phrases it is a separate concern, worth its own evaluator if
you care about tone.

The general principle: **a reference output should encode what must be true, at the loosest
granularity that still catches the bug.** You will meet it again in lesson 12, where the same
instinct applies to trajectories.

</details>

---

## 5. Your first evaluator

Start with a deterministic one. It is cheap, it is not itself a model that can be wrong, and it
catches the specific bug you found.

In [ ]:
def mentions_refund(text: str) -> bool:
    lowered = text.lower()
    return "refund" in lowered and "no refund" not in lowered and "cannot refund" not in lowered


def correct_refund_decision(outputs: dict, reference_outputs: dict) -> dict:
    """Fail when the agent offers a refund on a repair-only case."""
    answer = outputs.get("answer", "")
    expected = reference_outputs.get("expected_outcome")

    if expected == "repair_only":
        offered = mentions_refund(answer)
        return {
            "key": "correct_refund_decision",
            "score": 0.0 if offered else 1.0,
            "comment": "Offered a refund on a repair-only case." if offered else "Correctly withheld refund.",
        }

    return {"key": "correct_refund_decision", "score": 1.0, "comment": "Not a repair-only case."}

In [ ]:
def run_agent(inputs: dict) -> dict:
    """The thing under test. Takes dataset inputs, returns something evaluators can read."""
    result = agent.invoke({"messages": [{"role": "user", "content": inputs["question"]}]})
    return {"answer": result["messages"][-1].text}


experiment = client.evaluate(
    run_agent,
    data=DATASET_NAME,
    evaluators=[correct_refund_decision],
    experiment_prefix="baseline",
    max_concurrency=4,
)

print(experiment)

> 📸 **`10-experiment-view.png`** — The LangSmith experiment view showing seven rows, the correct_refund_decision column with a mix of 1.0 and 0.0 scores, and the aggregate score at the top.
>
> *Caption:* Per-example scores, and the aggregate that hides them.
>
> `https://raw.githubusercontent.com/langchain-samples/lc-colab-workshops/main/assets/screenshots/10-experiment-view.png`

---

## 6. Fix it, and prove the fix

Now the loop that makes all of this worth it. Change the prompt, re-run the **same** dataset,
compare experiments.

In [ ]:
STRICTER_PROMPT = (
    "You are a customer support agent for an office furniture retailer.\n"
    "Look up the order before answering, and note how many days ago it was delivered.\n"
    "Read the refund policy and apply it literally.\n"
    "\n"
    "CRITICAL: the 30-day boundary is measured from delivery.\n"
    "  - Faulty within 30 days -> refund or replacement is allowed.\n"
    "  - Faulty after 30 days  -> REPAIR ONLY. Do not offer a refund or replacement.\n"
    "  - Damaged on arrival    -> refund or replacement, no time limit.\n"
    "\n"
    "State the days-since-delivery and quote the policy line you applied."
)

fixed_agent = create_deep_agent(
    model=MODEL,
    tools=[lookup_order, search_tickets, get_refund_policy],
    system_prompt=STRICTER_PROMPT,
)


def run_fixed_agent(inputs: dict) -> dict:
    result = fixed_agent.invoke({"messages": [{"role": "user", "content": inputs["question"]}]})
    return {"answer": result["messages"][-1].text}


client.evaluate(
    run_fixed_agent,
    data=DATASET_NAME,
    evaluators=[correct_refund_decision],
    experiment_prefix="stricter-prompt",
    max_concurrency=4,
)

Open both experiments in LangSmith and compare them side by side. This is the artefact that
makes the difference between "I think it's better" and "here are the seven cases, and this one
regressed".

> 📸 **`10-compare-experiments.png`** — The LangSmith comparison view with baseline and stricter-prompt side by side, one row highlighted where the score changed between runs.
>
> *Caption:* Comparing two experiments over the same dataset. Read the rows, not the average.
>
> `https://raw.githubusercontent.com/langchain-samples/lc-colab-workshops/main/assets/screenshots/10-compare-experiments.png`

### 🧠 Checkpoint

Suppose the aggregate score rises from 0.71 to 0.86 — but one example that passed before now
fails.

Which view shows you that, and why does the aggregate hide it?

<details><summary>Show answer</summary>

The **comparison view**, row by row. The aggregate cannot show it, by construction: it is a mean,
and a mean is one number summarising seven independent facts. Two examples improving and one
regressing nets out positive.

This matters more than it sounds. Regressions are almost never uniform — a prompt change that
fixes a class of failures very often breaks a neighbouring one, because you sharpened a rule
that some other case relied on being fuzzy.

The habit worth forming: **the aggregate tells you whether to look; the rows tell you what
happened.** Any experiment you accept on the strength of its average alone is one you have not
actually read.

</details>

### ✍️ Exercise

Extend the suite with your own failures:

1. Run the agent on two or three questions of your own — try to find one it gets wrong.
2. Add each as an example, with a reference output that encodes the **decision**, not wording.
3. Write a second evaluator. Suggestion: *did the answer cite a specific policy line?*
4. Re-run both experiments and see whether your new evaluator agrees with the first one.

If your new examples all pass immediately, they are probably too easy. Aim for the boundaries —
the 30-day edge, the 14-day exchange window, the $200 approval threshold.

<details><summary>Show a solution</summary>

```python
def cites_policy(outputs: dict, reference_outputs: dict) -> dict:
    """The answer should quote or closely paraphrase the governing policy line."""
    answer = outputs.get("answer", "").lower()
    signals = ["policy", "within 30 days", "after 30 days", "no time limit",
               "restocking", "14 days", "business days"]
    hit = any(s in answer for s in signals)
    return {
        "key": "cites_policy",
        "score": 1.0 if hit else 0.0,
        "comment": "Cited policy." if hit else "No policy reference — answer is unsourced.",
    }


client.create_examples(
    dataset_id=dataset.id,
    examples=[
        {"inputs": {"question": "Order 1045 chair broke 40 days after delivery. Full refund?"},
         "outputs": {"expected_outcome": "repair_only",
                     "policy": "Faulty after 30 days: repair only."}},
        {"inputs": {"question": "Order 1042 is damaged and costs $429. Can I refund it immediately?"},
         "outputs": {"expected_outcome": "refund_needs_approval",
                     "policy": "Refunds above $200 require human approval."}},
    ],
)

client.evaluate(
    run_fixed_agent,
    data=DATASET_NAME,
    evaluators=[correct_refund_decision, cites_policy],
    experiment_prefix="two-evaluators",
    max_concurrency=4,
)
```

</details>

---

## 📌 Key takeaways

- Every useful eval suite starts as **one bug you found by hand**.
- A dataset is a bug report you can re-run — that is the entire value proposition.
- Traces are the cheapest source of realistic test cases, because they are real traffic.
- Store the **decision and the reason** as the reference, not an ideal sentence.
- Prefer a deterministic evaluator when one exists: it is cheap and cannot itself be wrong.
- Aggregates tell you whether to look; **rows tell you what happened**.
- One evaluator beats zero. Ship it before perfecting it.

---

## ➡️ Next

**[11 · Unit tests for agents](https://colab.research.google.com/github/langchain-samples/lc-colab-workshops/blob/main/notebooks/11_single_step_evals.ipynb)**

That evaluator judged the final answer. But when it fails, you still do not know *which step*
went wrong — which is what single-step evals are for.